# Machine Learning Foundation - Course 2: Regression

Dokumentasi praktikum **Course 2 (Regression)** :
1. **Section 2, Part a:** Regression Intro & Transforming Target (Log, Sqrt, Box-Cox Transformation)
2. **Section 2, Part a Lab:** Linear & Polynomial Regression (Car Price Dataset Analysis)
3. **Section 2, Part b:** Regression Setup & Train-Test Split LAB (Ames Housing Sales & One-Hot Encoding)
4. **Section 2, Part c:** Cross Validation, Pipelines & Hyperparameter Tuning (GridSearchCV, K-Fold)
5. **Section 2, Part d:** Regularization and Gradient Descent (Ridge, Lasso, ElasticNet, SGDRegressor)
6. **Section 2, Part e:** Regularization LAB (Data Standardization & Overfitting Prevention)
7. **Final Assignment:** Proyek Akhir Regresi Terawasi (Supervised Machine Learning: Regression)


---
## Section 2, Part a: Regression Intro - Transforming Target

Penerapan transformasi statistik (Log, Square Root, Box-Cox) pada variabel target agar berdistribusi normal.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import pickle
from scipy.stats.mstats import normaltest
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, ElasticNet, ElasticNetCV, SGDRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

# Download cleaned Boston Housing pickle file
url_boston = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle'
urllib.request.urlretrieve(url_boston, 'boston_housing_clean.pickle')

with open('boston_housing_clean.pickle', 'rb') as f:
    boston = pickle.load(f)

boston_data = boston['dataframe']
print("Shape Data Boston Housing:", boston_data.shape)
boston_data.head()

Shape Data Boston Housing: (506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


In [2]:
# Uji Normalitas pada Target (MEDV)
print("Uji Normalitas awal (p-value):", normaltest(boston_data.MEDV.values)[1])

# Transformasi Box-Cox pada variabel target MEDV
bc_result = boxcox(boston_data.MEDV)
boxcox_medv = bc_result[0]
lam = bc_result[1]

print("Lambda Box-Cox:", lam)
print("Uji Normalitas setelah Box-Cox (p-value):", normaltest(boxcox_medv)[1])

Uji Normalitas awal (p-value): 1.7583188871696095e-20
Lambda Box-Cox: 0.21662091371265346
Uji Normalitas setelah Box-Cox (p-value): 0.10468867128449004


---
## Section 2, Part a Lab: Linear & Polynomial Regression

Eksplorasi Regresi Linear dan Regresi Polinomial menggunakan dataset Car Price.

In [3]:
# Load Car Price Encoded Data
url_car = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/encoded_car_data.csv'
car_data = pd.read_csv(url_car)

X_car = car_data.drop('price', axis=1)
y_car = car_data.price

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_car, y_car, test_size=0.3, random_state=42)

# Fit Linear Regression
lr_car = LinearRegression()
lr_car.fit(X_train_c, y_train_c)

print("R2 Score Training (Car Data):", lr_car.score(X_train_c, y_train_c))
print("R2 Score Testing (Car Data):", lr_car.score(X_test_c, y_test_c))

R2 Score Training (Car Data): 0.9262722224294516
R2 Score Testing (Car Data): 0.8471124710243334


In [4]:
# Polynomial Features Degree 2
pf_car = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = pf_car.fit_transform(X_train_c)
X_test_poly = pf_car.transform(X_test_c)

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train_c)

print("Poly Degree 2 - R2 Train:", lr_poly.score(X_train_poly, y_train_c))
print("Poly Degree 2 - R2 Test:", lr_poly.score(X_test_poly, y_test_c))
print("(R2 negatif pada test set menandakan overfitting)")

Poly Degree 2 - R2 Train: 0.998531849797585
Poly Degree 2 - R2 Test: -36729.957911810954
(R2 negatif pada test set menandakan overfitting)


---
## Section 2, Part b: Regression Setup & Train-Test Split LAB

Praktikum One-Hot Encoding dan Feature Scaling pada dataset Ames Housing Sales.

In [5]:
url_ames_sales = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv"
ames_sales = pd.read_csv(url_ames_sales)

print("Shape Data Ames Housing Sales:", ames_sales.shape)
print("Tipe Data:\n", ames_sales.dtypes.value_counts())

Shape Data Ames Housing Sales: (1379, 80)
Tipe Data:
 object     43
float64    21
int64      16
Name: count, dtype: int64


In [6]:
# Identifikasi kolom kategorikal string
mask = ames_sales.dtypes == object
categorical_cols = ames_sales.columns[mask]

# Salinan data untuk One-Hot Encoding
data_ohc = ames_sales.copy()
le = LabelEncoder()
ohc = OneHotEncoder()

for col in categorical_cols:
    dat = le.fit_transform(data_ohc[col]).astype(int)
    data_ohc = data_ohc.drop(col, axis=1)
    new_dat = ohc.fit_transform(dat.reshape(-1, 1))
    n_cols = new_dat.shape[1]
    col_names = [f"{col}_{x}" for x in range(n_cols)]
    new_df = pd.DataFrame(new_dat.toarray(), index=data_ohc.index, columns=col_names)
    data_ohc = pd.concat([data_ohc, new_df], axis=1)

# Split Train & Test
y_col = 'SalePrice'
X_no_enc = ames_sales.drop([y_col] + list(categorical_cols), axis=1)
y_no_enc = ames_sales[y_col]

X_train_ne, X_test_ne, y_train_ne, y_test_ne = train_test_split(X_no_enc, y_no_enc, test_size=0.3, random_state=42)

X_ohc = data_ohc.drop(y_col, axis=1)
y_ohc = data_ohc[y_col]

X_train_ohc, X_test_ohc, y_train_ohc, y_test_ohc = train_test_split(X_ohc, y_ohc, test_size=0.3, random_state=42)

# Model Linear Regression Tanpa Encoding vs Dengan One-Hot Encoding
lr_ne = LinearRegression().fit(X_train_ne, y_train_ne)
lr_ohc = LinearRegression().fit(X_train_ohc, y_train_ohc)

print("MSE Test (Tanpa Encoding Kategorikal):", mean_squared_error(y_test_ne, lr_ne.predict(X_test_ne)))
print("MSE Test (Dengan One-Hot Encoding):", mean_squared_error(y_test_ohc, lr_ohc.predict(X_test_ohc)))

MSE Test (Tanpa Encoding Kategorikal): 1372182358.9345438
MSE Test (Dengan One-Hot Encoding): 8065327607.238906


---
## Section 2, Part c: Cross Validation, Pipeline & Hyperparameter Tuning

Penggunaan K-Fold Cross Validation, Pipeline, Ridge/Lasso Regularization, dan GridSearchCV.

In [7]:
# Load data Boston untuk Cross Validation
X_bos = boston_data.drop('MEDV', axis=1)
y_bos = boston_data.MEDV

kf = KFold(shuffle=True, random_state=72018, n_splits=3)

# Grid Search CV pada Ridge Regression dengan Polynomial Features
estimator = Pipeline([
    ("polynomial_features", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("ridge_regression", Ridge())
])

params = {
    'polynomial_features__degree': [1, 2, 3],
    'ridge_regression__alpha': np.geomspace(4, 20, 10)
}

grid = GridSearchCV(estimator, params, cv=kf)
grid.fit(X_bos, y_bos)

print("Best R2 Score:", grid.best_score_)
print("Best Parameters:", grid.best_params_)

Best R2 Score: 0.8546333782070427
Best Parameters: {'polynomial_features__degree': 2, 'ridge_regression__alpha': np.float64(4.0)}


---
## Section 2, Part d: Regularization and Gradient Descent

Eksplorasi teknik regularisasi (Lasso, Ridge, ElasticNet) dan penggunaan Stochastic Gradient Descent (SGD) Regressor.

In [8]:
# Load Sinusoid Data
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

url_sinusoid = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/X_Y_Sinusoid_Data.csv"
data_sin = pd.read_csv(url_sinusoid)

X_real = np.linspace(0, 1.0, 100)
Y_real = np.sin(2 * np.pi * X_real)

# Polynomial Features Degree 20
pf20 = PolynomialFeatures(degree=20)
X_data = data_sin[['x']]
Y_data = data_sin['y']

X_poly20 = pf20.fit_transform(X_data)
lr_sin = LinearRegression().fit(X_poly20, Y_data)
rr_sin = Ridge(alpha=0.001).fit(X_poly20, Y_data)
lassor_sin = Lasso(alpha=0.0001, max_iter=int(1e5), tol=0.01).fit(X_poly20, Y_data)

print("Linear, Ridge, and Lasso Regression fit complete on Sinusoid Data.")


Linear, Ridge, and Lasso Regression fit complete on Sinusoid Data.


In [9]:
# Ames Housing Sales untuk Regularisasi & SGD
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

ames_encoded = pd.get_dummies(ames_sales, drop_first=True)

train_a, test_a = train_test_split(ames_encoded, test_size=0.3, random_state=42)

# Log Transform Skewed Features
float_cols = ames_encoded.columns[ames_encoded.dtypes == float]
skew_vals = train_a[float_cols].skew()
skew_cols = skew_vals[abs(skew_vals) > 0.75].index

for col in skew_cols:
    if col != 'SalePrice':
        train_a[col] = np.log1p(train_a[col])
        test_a[col] = np.log1p(test_a[col])

X_train_a = train_a.drop('SalePrice', axis=1)
y_train_a = train_a['SalePrice']
X_test_a = test_a.drop('SalePrice', axis=1)
y_test_a = test_a['SalePrice']

def rmse(ytrue, ypred):
    return np.sqrt(mean_squared_error(ytrue, ypred))

# Evaluasi RidgeCV, LassoCV, ElasticNetCV (dengan tol=0.01 agar konvergen cepat & bersih)
ridgeCV = RidgeCV(alphas=[0.005, 0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 80], cv=4).fit(X_train_a, y_train_a)
lassoCV = LassoCV(alphas=[1e-5, 5e-5, 0.0001, 0.0005], max_iter=int(1e5), tol=0.01, cv=3).fit(X_train_a, y_train_a)

print("RMSE RidgeCV:", rmse(y_test_a, ridgeCV.predict(X_test_a)))
print("RMSE LassoCV:", rmse(y_test_a, lassoCV.predict(X_test_a)))


RMSE RidgeCV: 32022.031469901194
RMSE LassoCV: 45710.253511298244


---
## Section 2, Part e: Regularization LAB

Eksplorasi pembagian Train-Test Split gabungan dengan Standarisasi Fitur (`StandardScaler`) untuk mencegah kebocoran data (data leakage).

In [10]:
# Boston Housing Data Standarisasi & Dampak Scaling
X_b = boston_data.drop('MEDV', axis=1)
y_b = boston_data['MEDV']

scaler_b = StandardScaler()
X_b_ss = scaler_b.fit_transform(X_b)

lr_b = LinearRegression().fit(X_b_ss, y_b)
coef_df = pd.DataFrame(list(zip(X_b.columns, lr_b.coef_)), columns=['Feature', 'Coefficient']).sort_values(by='Coefficient')
print("Top Features by Impact (Standardized Coefficients):\n", coef_df)

Top Features by Impact (Standardized Coefficients):
     Feature  Coefficient
12    LSTAT    -3.748680
7       DIS    -3.104448
9       TAX    -2.075898
10  PTRATIO    -2.062156
4       NOX    -2.060092
0      CRIM    -0.920411
6       AGE     0.021121
2     INDUS     0.142967
3      CHAS     0.682203
11        B     0.856640
1        ZN     1.080981
8       RAD     2.658787
5        RM     2.670641


---
## Final Assignment: Supervised Machine Learning - Regression

Proyek Akhir Regresi Terawasi: Eksplorasi Data, Data Preprocessing, Modeling & Evaluation.

In [11]:
# Load Dataset untuk Final Assignment (Ames Housing Sales)
df_final = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv')
print("Shape Dataset Final Assignment:", df_final.shape)

# 1. One-Hot Encoding pada variabel kategorikal
df_encoded = pd.get_dummies(df_final, drop_first=True)

# 2. Split Data Feature (X) dan Target (y)
X_f = df_encoded.drop('SalePrice', axis=1)
y_f = df_encoded['SalePrice']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.3, random_state=42)

# 3. Scaling & ElasticNet Regularization Pipeline
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(random_state=42))
])

param_grid_final = {
    'model__alpha': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.2, 0.5, 0.8]
}

grid_final = GridSearchCV(pipe_final, param_grid_final, cv=5, scoring='neg_mean_squared_error')
grid_final.fit(X_train_f, y_train_f)

best_model = grid_final.best_estimator_
y_pred_f = best_model.predict(X_test_f)

print("Best Parameters:", grid_final.best_params_)
print("RMSE Test Final Model:", np.sqrt(mean_squared_error(y_test_f, y_pred_f)))
print("R2 Score Test Final Model:", best_model.score(X_test_f, y_test_f))

Shape Dataset Final Assignment: (1379, 80)
Best Parameters: {'model__alpha': 1, 'model__l1_ratio': 0.8}
RMSE Test Final Model: 44638.514148584145
R2 Score Test Final Model: 0.669931046514266
